# EchoFactory NB-02: Preprocessing & Feature Extraction
Extract 2 fitur dari raw .wav untuk STgram-MFN:
- **Branch 0**: Log-Mel Spectrogram (n_fft=1024)
- **Branch 1**: Tgram - Tangential gram (n_fft=512, resolusi temporal lebih halus)

**RAM Strategy**: Proses 1 mesin -> simpan .pt -> gc.collect() -> lanjut mesin berikutnya


In [ ]:
import os
import gc
import glob
import json
import numpy as np
import librosa
import torch
import torch.nn.functional as F
from tqdm import tqdm

print(f'torch {torch.__version__} | librosa {librosa.__version__}')


In [ ]:
DATASET_ROOT = '/kaggle/input/mimii-dataset'
OUT_DIR = '/kaggle/working'
TARGET_SNR = '0_dB'
MACHINE_TYPES = ['fan', 'pump', 'slider', 'valve']
SR = 16000
IMG_SIZE = (128, 128)

# Mel Spectrogram params
MEL_N_MELS = 128
MEL_N_FFT = 1024
MEL_HOP = 512

# Tgram params (FFT lebih kecil = temporal lebih detail)
TG_N_MELS = 128
TG_N_FFT = 512
TG_HOP = 512

os.makedirs(f'{OUT_DIR}/features', exist_ok=True)
print(f'Config OK | IMG_SIZE={IMG_SIZE} | SR={SR}')


In [ ]:
def compute_mel(wav):
    mel = librosa.feature.melspectrogram(
        y=wav, sr=SR, n_mels=MEL_N_MELS, n_fft=MEL_N_FFT, hop_length=MEL_HOP
    )
    return librosa.power_to_db(mel, ref=np.max)

def compute_tgram(wav):
    mel = librosa.feature.melspectrogram(
        y=wav, sr=SR, n_mels=TG_N_MELS, n_fft=TG_N_FFT, hop_length=TG_HOP
    )
    return librosa.power_to_db(mel, ref=np.max)

def normalize(x):
    return (x - x.mean()) / (x.std() + 1e-8)

def to_tensor(x_np):
    t = torch.FloatTensor(x_np).unsqueeze(0).unsqueeze(0)
    return F.interpolate(t, size=IMG_SIZE, mode='bilinear', align_corners=False).squeeze(0)

def extract_features(wav_path):
    wav = librosa.load(wav_path, sr=SR, mono=True)[0]
    mel_feat = to_tensor(normalize(compute_mel(wav)))
    tg_feat = to_tensor(normalize(compute_tgram(wav)))
    return torch.cat([mel_feat, tg_feat], dim=0)  # [2, 128, 128]

# Test
test_file = glob.glob(f'{DATASET_ROOT}/0_dB_fan/fan/id_00/normal/*.wav')[0]
feat = extract_features(test_file)
print(f'Feature shape: {feat.shape}  (expected: [2, 128, 128])')
print(f'Branch 0 (mel):   mean={feat[0].mean():.3f}, std={feat[0].std():.3f}')
print(f'Branch 1 (tgram): mean={feat[1].mean():.3f}, std={feat[1].std():.3f}')


In [ ]:
import matplotlib.pyplot as plt
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Fitur STgram-MFN: Dual Branch Input', fontsize=13, fontweight='bold')
a1.imshow(feat[0].numpy(), aspect='auto', origin='lower', cmap='magma')
a1.set_title('Branch 0: Mel Spectrogram (n_fft=1024)')
a1.set_xlabel('Time Frames')
a1.set_ylabel('Mel Bins')
a2.imshow(feat[1].numpy(), aspect='auto', origin='lower', cmap='viridis')
a2.set_title('Branch 1: Tgram (n_fft=512)')
a2.set_xlabel('Time Frames')
a2.set_ylabel('Mel Bins')
plt.tight_layout()
plt.savefig('/kaggle/working/feature_branches.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
label_maps = {}
for m in MACHINE_TYPES:
    mp = os.path.join(DATASET_ROOT, f'{TARGET_SNR}_{m}', m)
    if not os.path.exists(mp):
        continue
    ids = sorted(os.listdir(mp))
    label_maps[m] = {mid: i for i, mid in enumerate(ids)}
    print(f'  {m}: {label_maps[m]}')

with open('/kaggle/working/label_maps.json', 'w') as f:
    json.dump(label_maps, f, indent=2)
print('Label maps saved.')


In [ ]:
for machine in MACHINE_TYPES:
    print(f'\nProcessing {machine.upper()}...')
    lmap = label_maps.get(machine, {})
    mp = os.path.join(DATASET_ROOT, f'{TARGET_SNR}_{machine}', machine)
    if not os.path.exists(mp):
        print(f'  Path not found: {mp}')
        continue

    for condition in ['normal', 'abnormal']:
        all_feats, all_labels, all_paths = [], [], []

        for mid, label_id in lmap.items():
            cp = os.path.join(mp, mid, condition)
            if not os.path.exists(cp):
                continue
            files = sorted(glob.glob(os.path.join(cp, '*.wav')))
            print(f'  {mid}/{condition}: {len(files)} files')

            for fp in tqdm(files, desc=mid, leave=False):
                try:
                    all_feats.append(extract_features(fp))
                    all_labels.append(label_id)
                    all_paths.append(fp)
                except Exception as e:
                    print(f'  Skip {os.path.basename(fp)}: {e}')

        if not all_feats:
            continue

        save_path = f'/kaggle/working/features/{machine}_{condition}.pt'
        torch.save({
            'features': torch.stack(all_feats),
            'labels': torch.LongTensor(all_labels),
            'paths': all_paths
        }, save_path)
        mb = os.path.getsize(save_path) / (1024 ** 2)
        print(f'  SAVED: {save_path} | Shape: {torch.stack(all_feats).shape} | {mb:.1f} MB')
        del all_feats, all_labels, all_paths
        gc.collect()

    print(f'{machine.upper()} DONE')


In [ ]:
print('Verifikasi hasil preprocessing:')
for m in MACHINE_TYPES:
    for c in ['normal', 'abnormal']:
        p = f'/kaggle/working/features/{m}_{c}.pt'
        if os.path.exists(p):
            d = torch.load(p)
            n = d['features'].shape[0]
            mb = os.path.getsize(p) / (1024 ** 2)
            print(f'  {m:8s}/{c:8s}: N={n:4d} | {mb:.1f} MB')
        else:
            print(f'  {m}/{c}: TIDAK ADA')
print('\nNB-02 SELESAI! -> Lanjut ke NB-03_Train_STgramMFN.ipynb')
